# Revisions

Keep only pixels surrounded by all pixels of the same age on all sides

Make map illustrating the shape of secondary forest patches greater than 1ha of the same age


In [4]:
import ee
import geemap
from utils import *
ee.Authenticate(auth_mode='notebook')
ee.Initialize(project = 'extents-490617')

config = ProjectConfig()
roi = config.roi
data_folder = config.data_folder

In [5]:
# there are objects defined in scripts 1 - 8 that will be used here. This requires running them in this script:

import nbimporter # lets you import notebooks like regular modules

%run 3_grids.ipynb # run full script here to obtain inner_mask
%run objects.ipynb

# Neighbors of the same age

keep only pixels surrounded by all pixels of the same age

In [6]:
inner_patch = ee.Image(0)

proj = age.projection().getInfo()

age = age.updateMask(amazon_mask)
for year in range(1, 35):
    age_img = age.eq(year).unmask(0)

    sum_neighbors = age_img.focalMin(radius = 1, kernelType="square", units='pixels')

    inner_patch = inner_patch.add(sum_neighbors)

inner_patch = inner_patch.selfMask().reproject(age.projection())

# inner_mask is the mask including only secondary forest pixels in the interior of an ESA CCI cell (code in 3_grids.ipynb)
age_masked = age.updateMask(inner_mask).updateMask(inner_patch)

# create_grid(age_masked, region_name = "amazon", cell_size = 10000, file_name = "edge_removed_by_age")

## Same-age patches

Keep only patches of the same age and greater than 1ha

Do not exclude edge pixels. Selecting exclusively based on area and age.

Will obtain polygons looking like this:

![note](secondary_pixels_1ha.png)

In [ ]:
grid = amazon.geometry().coveringGrid('EPSG:4326', 100000)

grid_list = grid.toList(grid.size())
n = grid.size().getInfo()

for i in range(n):
    cell = ee.Feature(grid_list.get(i))

    vectors = age.reduceToVectors(
        geometry=cell.geometry(),
        geometryType='polygon',
        scale=30,
        eightConnected=True,
        maxPixels=1e12,
        labelProperty='age',
        tileScale = 16
    )

    vectors = vectors.map(
        lambda f: f.set({
            'area_m2': f.geometry().area(maxError=1),
            'tile_id': i + 1
        })
    ).filter(ee.Filter.gt('area_m2', 10000)).select(['age', 'tile_id'])

    task = ee.batch.Export.table.toAsset(
        collection=vectors,
        description=f"secondary_age_vectors_{i+1:03d}",
        assetId=f"{data_folder}/secondary_polygons/secondary_age_vectors_{i+1:03d}"
    )
    # task.start()

#309 needs to be run again

### Export age and biomass for ESA CCI for the same-age patches

In [ ]:
# for each collection, we will make a new collection by sampling one random pixel of age within it and returning it as a point feature.

def one_point_per_poly(f):
    geom = f.geometry()
    pt = ee.FeatureCollection.randomPoints(
        region = geom,
        points = 1,
        seed = ee.Number(1),   # or derive a deterministic seed if needed
        maxError = 1
    ).first()
    return ee.Feature(pt).copyProperties(f)

asset_list = ee.data.listAssets({'parent': f"{data_folder}/secondary_polygons"})['assets']

feature_ids = [a['name'] for a in asset_list]                    

for asset_id in feature_ids:
    polygons_fc = ee.FeatureCollection(asset_id)

    points_fc = ee.FeatureCollection(polygons_fc.map(one_point_per_poly))

    task = ee.batch.Export.table.toAsset(
        collection = points_fc,
        description = f"secondary_age_points_{asset_id[-3:]}",
        assetId = f"{data_folder}/secondary_points/secondary_age_vectors_{asset_id[-3:]}"
    )
    # task.start()
